<a href="https://colab.research.google.com/github/AhmadSyahreza/Katanya-kami-lomba/blob/main/Allnew.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# CELL 2: Config & imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import List

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

pd.set_option("display.max_columns", 50)
np.random.seed(42)
os.makedirs("plots", exist_ok=True)

@dataclass
class Config:
    date_col: str = "date"
    target_col: str = "y"
    freq: str = "MS"            # 'D' harian, 'MS' bulanan, 'YS' tahunan
    seasonality: int = 12       # m untuk MASE
    horizon: int = 1            # h langkah ke depan

    # split temporal
    train_end: str = "2022-12-31"
    val_end: str = "2023-12-31"

    # fitur
    lags: List[int] = field(default_factory=lambda: [1, 2, 3, 6, 12])
    roll_windows: List[int] = field(default_factory=lambda: [3, 6, 12])

    n_splits: int = 5

cfg = Config()
cfg

In [ ]:
# CELL 3: Load data
# >>> GANTI fungsi ini dengan pembacaan file kamu, contoh:
# df_raw = pd.read_csv("data.csv", parse_dates=[cfg.date_col])

def load_demo():
    rng = pd.date_range("2018-01-01", "2024-12-01", freq=cfg.freq)
    n = len(rng)
    t = np.arange(n)
    y = (100
         + 0.5 * t
         + 10 * np.sin(2 * np.pi * t / 12)
         + np.random.default_rng(42).normal(0, 3, n))
    df = pd.DataFrame({cfg.date_col: rng, cfg.target_col: y})
    df.loc[5:6, cfg.target_col] = np.nan    # simulasi missing
    df.loc[20, cfg.target_col] = 9999       # simulasi outlier
    return df

df_raw = load_demo()
df_raw.head()

In [ ]:
# CELL 4: Audit data mentah
def audit_basic(df, cfg):
    print("=" * 60)
    print("AUDIT DATA MENTAH")
    print("=" * 60)
    print(f"Shape            : {df.shape}")
    print(f"Kolom            : {list(df.columns)}")
    print(f"Tipe data        :\n{df.dtypes}")
    print(f"Missing per kolom:\n{df.isna().sum()}")
    print(f"Duplikat tanggal : {df[cfg.date_col].duplicated().sum()}")
    print(f"Range tanggal    : {df[cfg.date_col].min()} -> {df[cfg.date_col].max()}")
    print(f"Target describe  :\n{df[cfg.target_col].describe()}")
    print("=" * 60)

audit_basic(df_raw, cfg)

In [ ]:
# CELL 5: Cleaning
def clean(df, cfg):
    df = df.copy()
    df[cfg.date_col] = pd.to_datetime(df[cfg.date_col])
    df = df.sort_values(cfg.date_col).reset_index(drop=True)

    dup = df[cfg.date_col].duplicated().sum()
    if dup:
        print(f"[clean] hapus {dup} baris duplikat tanggal")
        df = df.drop_duplicates(subset=cfg.date_col, keep="last")

    # Reindex ke frekuensi penuh (tanggal hilang jadi NaN)
    df = df.set_index(cfg.date_col).asfreq(cfg.freq).reset_index()

    # Outlier: pakai ROLLING median (hanya ke belakang), bukan median global
    y = df[cfg.target_col]
    roll_med = y.rolling(window=cfg.seasonality, min_periods=4).median()
    roll_mad = (y - roll_med).abs().rolling(window=cfg.seasonality, min_periods=4).median()
    mask = (y - roll_med).abs() > 6 * roll_mad.replace(0, np.nan)
    print(f"[clean] outlier diganti rolling-median: {int(mask.sum())}")
    df[cfg.target_col] = y.mask(mask, roll_med)

    # Missing: interpolate HANYA ke arah depan (bukan bfill!)
    before = int(df[cfg.target_col].isna().sum())
    df[cfg.target_col] = df[cfg.target_col].interpolate(
        method="time", limit_direction="forward"
    )
    after = int(df[cfg.target_col].isna().sum())
    print(f"[clean] NaN target: {before} -> {after}")

    df = df.dropna(subset=[cfg.target_col]).reset_index(drop=True)
    return df

df = clean(df_raw, cfg)
df.head()

In [ ]:
# CELL 6: EDA
def run_eda(df, cfg):
    fig, ax = plt.subplots(3, 1, figsize=(12, 10))

    ax[0].plot(df[cfg.date_col], df[cfg.target_col], marker="o", lw=1)
    ax[0].set_title("Time Series Target")
    ax[0].grid(alpha=0.3)

    s = df.set_index(cfg.date_col)[cfg.target_col].asfreq(cfg.freq)
    try:
        dec = seasonal_decompose(s, model="additive", period=cfg.seasonality)
        ax[1].plot(dec.trend, label="trend")
        ax[1].plot(dec.seasonal, label="seasonal")
        ax[1].legend(); ax[1].set_title("Decomposition")
    except Exception as e:
        ax[1].text(0.5, 0.5, f"decompose gagal: {e}", ha="center")

    plot_acf(s.dropna(), ax=ax[2], lags=min(24, len(s)//2 - 1))
    ax[2].set_title("ACF")

    plt.tight_layout()
    plt.savefig("plots/eda.png", dpi=110)
    plt.show()

    tmp = df.copy()
    tmp["month"] = pd.to_datetime(tmp[cfg.date_col]).dt.month
    print("[eda] rata-rata per bulan:")
    print(tmp.groupby("month")[cfg.target_col].mean().round(2))

run_eda(df, cfg)

In [ ]:
# CELL 7: Fitur & target
def make_features(df, cfg):
    df = df.copy()
    y = df[cfg.target_col]

    for lag in cfg.lags:
        if lag < cfg.horizon:
            raise ValueError(f"Lag {lag} < horizon {cfg.horizon} -> LEAKAGE")
        df[f"lag_{lag}"] = y.shift(lag)

    # Rolling: shift(horizon) dulu, baru rolling
    for w in cfg.roll_windows:
        df[f"rollmean_{w}"] = y.shift(cfg.horizon).rolling(w).mean()
        df[f"rollstd_{w}"]  = y.shift(cfg.horizon).rolling(w).std()

    # Momentum
    df["diff_1"]       = y.shift(cfg.horizon) - y.shift(cfg.horizon + 1)
    df["pct_change_1"] = y.shift(cfg.horizon).pct_change()

    # Kalender (deterministik, aman)
    d = pd.to_datetime(df[cfg.date_col])
    df["month"]   = d.dt.month
    df["quarter"] = d.dt.quarter
    df["year"]    = d.dt.year

    # Target: y(t+h)
    df["y_target"] = y.shift(-cfg.horizon)

    df = df.dropna().reset_index(drop=True)
    return df

df_feat = make_features(df, cfg)
print("Shape fitur:", df_feat.shape)
df_feat.head()

In [ ]:
# CELL 8: Audit leakage fitur
def audit_features(df_feat, cfg, max_corr=0.95):
    print("=" * 60)
    print("AUDIT LEAKAGE FITUR")
    print("=" * 60)

    y = df_feat["y_target"]
    drop = [cfg.target_col, "y_target", cfg.date_col]
    X = df_feat.drop(columns=drop, errors="ignore")

    for col in X.columns:
        if X[col].equals(y):
            print(f"  [!!] '{col}' identik dengan target -> LEAKAGE LANGSUNG")

    corr_y = X.corrwith(y).abs().sort_values(ascending=False)
    print("\n  Korelasi fitur vs target (top 10):")
    for col, v in corr_y.head(10).items():
        flag = "  <-- CURIGA (>0.95)" if v > max_corr else ""
        print(f"    {col:18s}: {v:.4f}{flag}")

    y_future = df_feat[cfg.target_col].shift(-1)
    corr_f = X.corrwith(y_future).abs().sort_values(ascending=False)
    print("\n  Fitur paling berkorelasi dengan y(t+1):")
    for col, v in corr_f.head(5).items():
        flag = "  <-- CURIGA: tahu masa depan?" if v > 0.99 else ""
        print(f"    {col:18s}: {v:.4f}{flag}")

    print("\n  Cek lag vs horizon:")
    for col in X.columns:
        if col.startswith("lag_"):
            lag_val = int(col.split("_")[1])
            ok = "ok " if lag_val >= cfg.horizon else "[!!] LEAKAGE"
            print(f"    {ok} '{col}' lag={lag_val} horizon={cfg.horizon}")
    print("=" * 60)

audit_features(df_feat, cfg)

In [ ]:
# CELL 9: Split temporal (train / val / test)
def temporal_split(df, cfg):
    df = df.copy()
    df[cfg.date_col] = pd.to_datetime(df[cfg.date_col])
    train = df[df[cfg.date_col] <= cfg.train_end].reset_index(drop=True)
    val   = df[(df[cfg.date_col] > cfg.train_end) & (df[cfg.date_col] <= cfg.val_end)].reset_index(drop=True)
    test  = df[df[cfg.date_col] > cfg.val_end].reset_index(drop=True)
    print(f"[split] train={len(train)}  val={len(val)}  test={len(test)}")
    return train, val, test

train, val, test = temporal_split(df_feat, cfg)

X_cols = [c for c in train.columns
          if c not in [cfg.date_col, cfg.target_col, "y_target"]]

X_train, y_train = train[X_cols], train["y_target"]
X_val,   y_val   = val[X_cols],   val["y_target"]
X_test,  y_test  = test[X_cols],  test["y_target"]

# y_train_raw dipakai untuk denominator MASE
y_train_raw = train[cfg.target_col].values
y_val_raw   = val[cfg.target_col].values
y_test_raw  = test[cfg.target_col].values

X_cols

In [ ]:
# CELL 10: Metrik
def mase(y_true, y_pred, y_train, m=12):
    y_true  = np.asarray(y_true, float)
    y_pred  = np.asarray(y_pred, float)
    y_train = np.asarray(y_train, float)

    if m <= 1:
        naive_err = np.abs(np.diff(y_train))
    else:
        naive_err = np.abs(y_train[m:] - y_train[:-m])

    scale = np.mean(naive_err)
    if not np.isfinite(scale) or scale == 0:
        raise ValueError("MASE scale 0/NaN. Cek data & seasonality.")

    return np.mean(np.abs(y_true - y_pred)) / scale

def smape(y_true, y_pred):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    denom = np.where(denom == 0, 1e-9, denom)
    return 100 * np.mean(np.abs(y_true - y_pred) / denom)

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def report(y_true, y_pred, y_train, m, label=""):
    print(f"[{label:20s}] "
          f"MAE={mean_absolute_error(y_true, y_pred):7.3f}  "
          f"RMSE={rmse(y_true, y_pred):7.3f}  "
          f"MASE={mase(y_true, y_pred, y_train, m):6.4f}  "
          f"sMAPE={smape(y_true, y_pred):6.2f}%")

In [ ]:
# CELL 11: Baseline walk-forward
def evaluate_baselines(y_train_raw, y_val_raw, cfg):
    print("=" * 60)
    print("BASELINE (hanya pakai data masa lalu)")
    print("=" * 60)

    hist = list(y_train_raw)
    p_naive, p_snaive, p_drift = [], [], []

    for y_true in y_val_raw:
        p_naive.append(hist[-1])
        p_snaive.append(hist[-cfg.seasonality])
        p_drift.append(hist[-1] + cfg.horizon*(hist[-1]-hist[0])/(len(hist)-1))
        hist.append(y_true)

    for name, preds in [("naive", p_naive),
                        ("seasonal_naive", p_snaive),
                        ("drift", p_drift)]:
        report(y_val_raw, preds, y_train_raw, cfg.seasonality, label=name)
    print("=" * 60)

evaluate_baselines(y_train_raw, y_val_raw, cfg)

In [ ]:
# CELL 12: Model
def get_models():
    return {
        "ridge": Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc",  StandardScaler()),
            ("m",   Ridge(alpha=1.0)),
        ]),
        "rf": Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("m",   RandomForestRegressor(
                n_estimators=400, max_depth=6, min_samples_leaf=3,
                random_state=42, n_jobs=-1)),
        ]),
    }

results = {}
for name, model in get_models().items():
    model.fit(X_train, y_train)
    pred_val = model.predict(X_val)
    m = mase(y_val.values, pred_val, y_train_raw, cfg.seasonality)
    report(y_val, pred_val, y_train_raw, cfg.seasonality, label=f"val/{name}")
    results[name] = (m, model)

best_name = min(results, key=lambda k: results[k][0])
print(f"\n>> Model terbaik (berdasarkan VAL): {best_name} "
      f"(MASE={results[best_name][0]:.4f})")

In [ ]:
# CELL 13: Evaluasi akhir di TEST
best_model = results[best_name][1]
pred_test = best_model.predict(X_test)

report(y_test, pred_test, y_train_raw, cfg.seasonality, label=f"TEST/{best_name}")

# Diagnostik residual
res = y_test.values - pred_test
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(test[cfg.date_col], y_test.values, marker="o", label="actual")
ax[0].plot(test[cfg.date_col], pred_test, marker="x", label="pred")
ax[0].legend(); ax[0].set_title("Actual vs Pred (TEST)")
ax[1].scatter(pred_test, res, alpha=0.7); ax[1].axhline(0, color="r")
ax[1].set_title("Residual vs Pred")
ax[2].hist(res, bins=15); ax[2].set_title("Distribusi Residual")
plt.tight_layout()
plt.savefig("plots/diagnostics.png", dpi=110)
plt.show()

print(f"mean residual = {res.mean():.3f} (idealnya ~0)")
print(f"std  residual = {res.std():.3f}")

In [ ]:
# CELL 14: Walk-forward CV untuk stabilitas skor
from sklearn.model_selection import TimeSeriesSplit

def walk_forward_cv(df_feat, cfg, model):
    tscv = TimeSeriesSplit(n_splits=cfg.n_splits, gap=cfg.horizon)
    X = df_feat[X_cols].values
    y = df_feat["y_target"].values
    y_raw = df_feat[cfg.target_col].values

    scores = []
    for i, (tr, va) in enumerate(tscv.split(X)):
        m = model
        m.fit(X[tr], y[tr])
        p = m.predict(X[va])
        s = mase(y[va], p, y_raw[tr], cfg.seasonality)
        scores.append(s)
        print(f"  fold {i+1}: MASE={s:.4f}")

    print(f"\n  mean MASE = {np.mean(scores):.4f}  std = {np.std(scores):.4f}")
    return scores

walk_forward_cv(df_feat, cfg, get_models()["ridge"])

In [ ]:
# CELL 15: Checklist
checklist = {
    "Split temporal (bukan random)": True,
    "Gap = horizon di CV":           cfg.horizon >= 1,
    "Semua lag >= horizon":          all(l >= cfg.horizon for l in cfg.lags),
    "Rolling di-shift(horizon)":     True,   # lihat make_features
    "Imputer fit di train":          True,   # Pipeline menjamin
    "Scaler fit di train":           True,
    "MASE denom dari train":         True,
    "Test disentuh sekali":          True,
    "Ada baseline naive":            True,
    "Ada audit leakage fitur":       True,
}

for k, v in checklist.items():
    print(f"  [{'v' if v else 'x'}] {k}")